# Tagging Proof of Concept

This notebook aims to emulate the process of passing a regulatory document through a PDF parser (e.g. LlamaParse) and then using the appropriate prompts to extract tagging information from the textual content generated from said document.

The ultimate aim is to have this process running as an API endpoint, with a wrapper site accessing said endpoint to be created at a future date. Future plans also include integrations with the BackBlaze document database and SupaBase for textual information, as well as creating a RAG application using said documents.

In [11]:
# imports
import os
import logging
from pathlib import Path
from dotenv import load_dotenv

from llama_cloud_services import LlamaParse

In [12]:
# Load env & set configs
# --- Load Environment Variables from .env file ---
load_dotenv()
# ---

# --- Basic Logging Setup ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [82]:
DOC_NAME = 'PSE_IRP_ElectricChapters_3.31.2025.pdf'

## Parsing: LlamaParse

In [15]:
result = await LlamaParse().aparse('./PSE_IRP_ElectricChapters_3.31.2025.pdf')

2025-04-22 12:04:08,246 - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 200 OK"


Started parsing the file under job_id 44057c0b-93f3-4d81-bc00-c8c594623fc4


2025-04-22 12:04:09,487 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/44057c0b-93f3-4d81-bc00-c8c594623fc4 "HTTP/1.1 200 OK"
2025-04-22 12:04:11,617 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/44057c0b-93f3-4d81-bc00-c8c594623fc4 "HTTP/1.1 200 OK"
2025-04-22 12:04:14,775 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/44057c0b-93f3-4d81-bc00-c8c594623fc4 "HTTP/1.1 200 OK"
2025-04-22 12:04:18,976 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/44057c0b-93f3-4d81-bc00-c8c594623fc4 "HTTP/1.1 200 OK"
2025-04-22 12:04:24,395 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/44057c0b-93f3-4d81-bc00-c8c594623fc4 "HTTP/1.1 200 OK"
2025-04-22 12:04:29,678 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/44057c0b-93f3-4d81-bc00-c8c594623fc4 "HTTP/1.1 200 OK"
2025-04-22 12:04:35,050 - INFO - HTTP Request: GET https://api.cloud.llamain

.

2025-04-22 12:04:56,314 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/44057c0b-93f3-4d81-bc00-c8c594623fc4 "HTTP/1.1 200 OK"
2025-04-22 12:05:01,593 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/44057c0b-93f3-4d81-bc00-c8c594623fc4 "HTTP/1.1 200 OK"
2025-04-22 12:05:06,971 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/44057c0b-93f3-4d81-bc00-c8c594623fc4 "HTTP/1.1 200 OK"
2025-04-22 12:05:07,372 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/44057c0b-93f3-4d81-bc00-c8c594623fc4/result/json "HTTP/1.1 200 OK"


In [16]:
documents = result.get_text_documents(split_by_page=True)
print(documents[0].text)

  2023 ELECTRIC
PROGRESS REPORT
     CHAPTERS 1–9
           PSE PUGET SOUND ENERGY


In [19]:
print(documents[100].text)

CHAPTER FIVE: KEY ANALYTICAL ASSUMPTIONS
estimates to each resource type based on the spur line length needed to interconnect each generic resource to the
transmission grid to account for this omission.
We expect generic resource capital costs to decline as technology advances push costs down. The declining cost
curves applied to different resource alternatives come from the 2022 ATB. The 2022 ATB provides three cost curves
for each resource: low, mid, and constant technology cost scenarios. We selected the mid-technology cost scenario for
the IRP cost curves, representing the most likely future cost projection.
We sourced generic resource O&M costs from the 2022 ATB for all generic resource technologies except thermal
technologies. We sourced generic CCCT and frame peaker fixed O&M from averaging our existing costs, as reported
in the 2021 FERC Form 1s. We adopted the fixed O&M that were reported for the Port Westward 2 facility as the
generic reciprocating peaker fixed O&M.¹⁴ We adop

## Creating Embeddings and Indexes
We first try using Cohere's `embed` API with the text nodes, passing into a simple vector index (FAISS).

In [66]:
from llama_index.core.node_parser import SentenceSplitter

text_nodes = await result.aget_text_nodes()
full_text = text_nodes[0].text

splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
chunks = splitter.split_text(full_text)

In [69]:
len(chunks)

269

In [68]:
len(chunks[0])

2619

In [70]:
import cohere
import time

co = cohere.Client(os.environ["COHERE_API_KEY"])

def batch_embed(texts, batch_size=100, delay_secs=10):
    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        try:
            response = co.embed(
                texts=batch,
                model="embed-english-v3.0",
                input_type="search_document"
            )
            embeddings.extend(response.embeddings)
        
        except cohere.errors.TooManyRequestsError as e:
            print(f"Rate limit hit, waiting before retrying: {e}")
            time.sleep(60)
            continue

        time.sleep(delay_secs)
    
    return embeddings

tagging_embeddings = batch_embed(chunks)

# tagging_embeddings = co.embed(
#     texts=chunks,
#     model="embed-english-v3.0",
#     input_type="classification"
# ).embeddings

Rate limit hit, waiting before retrying: status_code: 429, body: {'message': 'trial token rate limit exceeded, limit is 100000 tokens per minute'}


In [71]:
import faiss
import numpy as np

index = faiss.IndexFlatL2(len(tagging_embeddings[0]))
index.add(np.array(tagging_embeddings))

In [72]:
len(tagging_embeddings[0])

1024

## Testing Retrieval for Tagging

### Overview

In [ ]:
overview_query = f"""Using the document (named {DOC_NAME}), find the following fields, listed below. They will be listed as follows: 'Field Name (additional explanation)'.
Output each answer as follows: 'Field Name: answer', using the provided field name, with each on a new line.
If you don't know or if there isn't enough information, just put 'unknown'. Here are the fields:
1. Document Title (Full title of the document. Include subtitles, if they are present.)
2. Published Date (When the document was officially published. Answer in format MM/DD/YYYY. Use the document name if the information is not in the info provided.)
3. Org/Utility Name (Primary utility organization associated with document. Examples may be: Pacific Gas & Electric.)
4. Docket Number (Official proceeding identifier.)
5. Document Type (Primary classification of document. Examples may include: Rate Case, Compliance Filing, Resource Plan, Investigation, etc. These should be 1-2 words long.)
6. File format (Type of file. Use the document name.)"""

In [ ]:
overview_query_embedding = co.embed(
    texts=[overview_query],
    model="embed-english-v3.0",
    input_type="search_query"
).embeddings[0]

In [ ]:
D, I = index.search(np.array([overview_query_embedding]), k=10)
top_chunks = [chunks[i] for i in I[0]]

In [86]:
top_chunks

['This 2023 Electric Report is a planning exercise that evaluates how PSE will meet customer electric supply\nneeds. The analysis considers policies, costs, changing economic conditions, and the existing energy system to develop\na plan to meet the needs of our customers at the lowest reasonable cost over the next 20+ years.\nThroughout the resource planning process for this report, we focused on the following key objectives, which lay the\nfoundation for this and all future resource plans:\n    •   Build a reliable, diversified power portfolio of non-emitting resources\n    •   Ensure an equitable clean energy transition for all PSE customers\n    •   Ensure resource adequacy while delivering a clean energy transition\n    •   Ensure resource planning aligns with PSE’s Clean Energy Implementation Plan (CEIP) to meet our interim\n        targets and CETA obligations\nRecognizing that the 2023 Electric Report does not make resource or program implementation decisions is important.\nThe 

In [87]:
len(top_chunks[0])

2520

### Content Classification

In [89]:
content_query = f"""Using the document (named {DOC_NAME}), find the following fields, listed below. They will be listed as follows: 'Field Name (additional explanation)'.
Output each answer as follows: 'Field Name: answer', using the provided field name, with each on a new line.
If you don't know or if there isn't enough information, just put 'unknown'. Here are the fields:
1. Rate Impact (Does the document discuss impacts on energy bills? Only return Y for yes, N for no, or otherwise return unknown.)
2. Energy Resources (Which sources of energy do the document primarily discuss? Examples might include Solar, Battery Storage, Natural Gas, or Thermal Energy. Return the answers, separated by commas.)
3. Customer Classes (What types of customers are discussed? Examples might include Residential, Commercial, or Industrial. Return the answers, separated by commas.)
4. Physical Climate Risk (Is the primay purpose of the document discussing physical climate risk (wildfires, heat, etc.)? Return Y for yes, N for no, or otherwise return unknown.)
"""

In [90]:
content_query_embedding = co.embed(
    texts=[content_query],
    model="embed-english-v3.0",
    input_type="search_query"
).embeddings[0]

In [91]:
Dc, Ic = index.search(np.array([content_query_embedding]), k=10)
top_content_chunks = [chunks[i] for i in Ic[0]]

In [92]:
top_content_chunks

["If PSE relies on a certain amount of load reduction from demand response to handle a\npeak event, but customers opt out, we must use generating resources to fill the customer’s needs.\nWe organized demand response programs modeled for this 2023 Electric Report in four categories:\n   •   Behavioral Demand Response\n   •   Commercial and Industrial (C&I) Curtailment\n   •   Direct Load Control (DLC)\n   •   Dynamic Pricing or Critical Peak Pricing (CPP)\n                                              See Appendix E: Conservation Potential and Demand Response Assessments for the full\n                                              CPA Assessment. We included a complete discussion of how we chose the demand response\n                                                 programs for the preferred portfolio in Chapter Three: Resource Plan Decisions.\nFigure 2.4 lists the estimated 10-year achievable technical potential for demand response programs modeled for this\nreport's residential, commerc

## Use Cohere LLM to answer questions

In [ ]:
context = "\n\n".join(top_chunks)

response = co.chat(
    model="command-r",
    message=f"Answer the question based on the following context:\n\n{context}\n\nQuestion: {overview_query}\nAnswer:",
    max_tokens=300,
    temperature=0
)

print(response.text.strip())

2025-04-22 16:04:02,921 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


Document Title: 2023 Electric Progress Report
Published Date: 03/31/2023
Org/Utility Name: Puget Sound Energy
Docket Number: unknown
Document Type: Resource Plan
File format: PDF


In [94]:
content_context = "\n\n".join(top_content_chunks)

response = co.chat(
    model="command-r",
    message=f"Answer the question based on the following context:\n\n{content_context}\n\nQuestion: {content_query}\nAnswer:",
    max_tokens=300,
    temperature=0
)

print(response.text.strip())

2025-04-22 16:16:09,524 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


1. Rate Impact: Y

2. Energy Resources: Solar, Wind, Battery Storage, Hydro, Natural Gas, Hydrogen, Biomass

3. Customer Classes: Residential, Commercial, Industrial

4. Physical Climate Risk: N
